#Dataset : [Global mining footprint mapped from high-resolution satellite imagery by 'Zenodo'](https://zenodo.org/records/6806817)

The dataset comprises 74,548 polygons, covering ~66,000 km2 of features like waste rock dumps, pits, water ponds, tailings dams, heap leach pads and processing/milling infrastructure.

**In this we are going to filter out the data for India only and discard everything else**

In [1]:
# =====================================================================================
#  INDIA MINING FOOTPRINT DATASET  -  ONE-CELL COLAB VERSION
#  Paste this whole cell into Google Colab and run it. It will:
#    1) download the Tang & Werner global mining polygons (Zenodo) and unpack them
#    2) keep only India, download matching Sentinel-2 images, build masks
#    3) save a Hugging Face dataset (train/val/test) + plain PNG images/masks
#    4) zip everything into ONE file and download it
#  Labels: 0 = background, 1 = mining.  Images: 512x512 RGB PNG, 10 m/pixel.
# =====================================================================================

# ---------------------------- EDIT THESE IF YOU WANT ----------------------------------
LIMIT = 200                              # number of mining chips to build. None = everything (hours!)
DATE_RANGE = "2019-11-01/2020-04-30"     # Sentinel-2 dates (India's dry season = fewer clouds)
WORKERS = 8                              # parallel downloads
SAVE_TO_DRIVE = False                    # True = keep results in Google Drive too (survives disconnects)
ZENODO_RECORD = "6806817"                # Tang & Werner global mining footprint
# ---------------------------------------------------------------------------------------

import importlib.util, subprocess, sys

# install anything missing (fresh Colab runtimes lack a few packages)
_need = {"geopandas": "geopandas", "shapely": "shapely", "pyproj": "pyproj", "rasterio": "rasterio",
         "pystac_client": "pystac-client", "planetary_computer": "planetary-computer", "PIL": "pillow",
         "tqdm": "tqdm", "datasets": "datasets", "pyogrio": "pyogrio", "requests": "requests",
         "pandas": "pandas", "numpy": "numpy"}
_missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

import os

# Faster remote GeoTIFF reads (must be set before rasterio is used)
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("GDAL_HTTP_MAX_RETRY", "3")
os.environ.setdefault("GDAL_HTTP_RETRY_DELAY", "2")
os.environ.setdefault("VSI_CACHE", "TRUE")
os.environ.pop("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", None)   # breaks signed URLs; make sure it is NOT set

import argparse
import hashlib
import json
import math
import random
import sys
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import shapely
from PIL import Image
from pyproj import Transformer
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.windows import from_bounds as window_from_bounds
from shapely.geometry import Point, box
from shapely.prepared import prep
from tqdm import tqdm

NE_URL = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
PLAN_CRS = "EPSG:7755"          # WGS 84 / India NSF LCC: low distortion over India, metres
PIXEL_M = 10                    # Sentinel-2 RGB/NIR resolution
SCL_CLOUD_CLASSES = [3, 8, 9, 10]   # cloud shadow, cloud medium, cloud high, thin cirrus


# ----------------------------------------------------------------------------
# 1. Load boundary + mining polygons, keep only India
# ----------------------------------------------------------------------------
def load_india(path):
    path = Path(path)
    if not path.exists():
        import urllib.request
        print(f"Downloading Natural Earth country boundaries to {path} ...")
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(NE_URL, path)
    gdf = gpd.read_file(path)
    col = next((c for c in ("ADMIN", "NAME", "admin", "name") if c in gdf.columns), None)
    if col is None:
        raise SystemExit(f"Can't find a country-name column in {path}. Columns: {list(gdf.columns)}")
    india = gdf[gdf[col] == "India"].to_crs(4326)
    if india.empty:
        raise SystemExit(f"No row with {col} == 'India' in {path}")
    return india


def load_mining(paths, india_geom, layer=None):
    frames = []
    for p in paths:
        print(f"Reading {p} ...")
        g = gpd.read_file(p, layer=layer) if layer else gpd.read_file(p)
        if g.crs is None:
            print(f"  WARNING: {p} has no CRS, assuming EPSG:4326")
            g = g.set_crs(4326)
        g = g[["geometry"]].to_crs(4326)
        print(f"  {len(g):,} features")
        frames.append(g)
    m = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry", crs=4326)
    m = m[~m.geometry.isna() & ~m.geometry.is_empty]
    m["geometry"] = m.geometry.make_valid()
    m = m.explode(index_parts=False, ignore_index=True)
    m = m[m.geom_type == "Polygon"]
    m = m[m.intersects(india_geom)].reset_index(drop=True)
    print(f"Mining polygons inside/touching India: {len(m):,}")
    if m.empty:
        raise SystemExit("No mining polygons fall inside India. Check your input files.")
    return m


# ----------------------------------------------------------------------------
# 2. Plan chip locations
# ----------------------------------------------------------------------------
def plan_mine_chips(mining, chip_m, dedupe_m):
    """One chip centred on each small mine; a grid of chips over big mines.
    Chips whose centre lands inside an already planned chip are skipped."""
    m = mining.to_crs(PLAN_CRS)
    order = np.argsort(-m.area.values)          # big mines first
    grid = defaultdict(list)

    def near(x, y):
        gx, gy = int(x // dedupe_m), int(y // dedupe_m)
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for (cx, cy) in grid[(gx + dx, gy + dy)]:
                    if abs(cx - x) < dedupe_m and abs(cy - y) < dedupe_m:
                        return True
        return False

    centers = []
    for i in order:
        geom = m.geometry.iloc[i]
        minx, miny, maxx, maxy = geom.bounds
        w, h = maxx - minx, maxy - miny
        if w <= 0.8 * chip_m and h <= 0.8 * chip_m:
            cands = [((minx + maxx) / 2, (miny + maxy) / 2)]
        else:
            nx, ny = math.ceil(w / chip_m), math.ceil(h / chip_m)
            cands = []
            for a in range(nx):
                for b in range(ny):
                    cx, cy = minx + (a + 0.5) * chip_m, miny + (b + 0.5) * chip_m
                    if geom.intersects(box(cx - chip_m / 2, cy - chip_m / 2, cx + chip_m / 2, cy + chip_m / 2)):
                        cands.append((cx, cy))
        for (cx, cy) in cands:
            if not near(cx, cy):
                grid[(int(cx // dedupe_m), int(cy // dedupe_m))].append((cx, cy))
                centers.append((cx, cy))

    tf = Transformer.from_crs(PLAN_CRS, "EPSG:4326", always_xy=True)
    lons, lats = tf.transform([c[0] for c in centers], [c[1] for c in centers])
    return [dict(kind="mine", lon=float(lo), lat=float(la)) for lo, la in zip(lons, lats)]


def plan_negative_chips(india_geom, mining, n, rng, clear_deg=0.07):
    """Random India locations with no mining polygon nearby (background examples)."""
    if n <= 0:
        return []
    sindex = mining.sindex
    pg = prep(india_geom)
    minx, miny, maxx, maxy = india_geom.bounds
    out, tries = [], 0
    while len(out) < n and tries < n * 500:
        tries += 1
        lon, lat = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        p = Point(lon, lat)
        if not pg.contains(p):
            continue
        if len(sindex.query(p.buffer(clear_deg))) > 0:
            continue
        out.append(dict(kind="neg", lon=lon, lat=lat))
    return out


def assign_split(lon, lat):
    """Spatial split on 0.5 degree cells so neighbouring chips never straddle splits."""
    key = f"{math.floor(lat / 0.5)}_{math.floor(lon / 0.5)}"
    h = int(hashlib.md5(key.encode()).hexdigest(), 16) % 1000 / 1000
    return "train" if h < 0.70 else ("val" if h < 0.85 else "test")


# ----------------------------------------------------------------------------
# 3. Fetch Sentinel-2 chip (Microsoft Planetary Computer)
# ----------------------------------------------------------------------------
_catalog = None
_catalog_lock = threading.Lock()


def get_catalog():
    global _catalog
    with _catalog_lock:
        if _catalog is None:
            import planetary_computer as pc
            from pystac_client import Client
            _catalog = Client.open(STAC_URL, modifier=pc.sign_inplace)
        return _catalog


def item_crs(item):
    p = item.properties
    if p.get("proj:code"):
        return CRS.from_string(p["proj:code"])
    if p.get("proj:epsg"):
        return CRS.from_epsg(int(p["proj:epsg"]))
    with rasterio.open(item.assets["B04"].href) as src:
        return src.crs


def chip_bounds(lon, lat, crs, size):
    """Chip bounds in the scene's CRS, snapped to the 10 m grid."""
    tf = Transformer.from_crs("EPSG:4326", crs.to_wkt(), always_xy=True)
    cx, cy = tf.transform(lon, lat)
    half = size * PIXEL_M / 2
    x0 = round((cx - half) / PIXEL_M) * PIXEL_M
    y0 = round((cy - half) / PIXEL_M) * PIXEL_M
    return (x0, y0, x0 + size * PIXEL_M, y0 + size * PIXEL_M)


def read_window(href, bounds, size, tries=3):
    last = None
    for k in range(tries):
        try:
            with rasterio.open(href) as src:
                win = window_from_bounds(*bounds, transform=src.transform)
                return src.read(1, window=win, out_shape=(size, size), boundless=True,
                                fill_value=0, resampling=Resampling.nearest)
        except Exception as e:  # network hiccup: back off and retry
            last = e
            time.sleep(1.5 * (k + 1))
    raise last


def to_reflectance(arr, asset):
    rb = (asset.extra_fields.get("raster:bands") or [{}])[0]
    return arr.astype("float32") * rb.get("scale", 1e-4) + rb.get("offset", 0.0)


def fetch_chip(lon, lat, args):
    """Return dict(rgb, crs, bounds, date, cloud, item_id) for the clearest scene, or None."""
    size = args.chip_px
    search = get_catalog().search(
        collections=["sentinel-2-l2a"],
        intersects={"type": "Point", "coordinates": [lon, lat]},
        datetime=args.date_range,
        query={"eo:cloud_cover": {"lt": args.max_scene_cloud}},
    )
    items = sorted(search.items(), key=lambda it: it.properties.get("eo:cloud_cover", 100))
    for item in items[:args.max_scenes_tried]:
        crs = item_crs(item)
        bounds = chip_bounds(lon, lat, crs, size)
        bands = []
        for b in ("B04", "B03", "B02"):  # R, G, B
            raw = read_window(item.assets[b].href, bounds, size)
            bands.append((raw, to_reflectance(raw, item.assets[b])))
        if (bands[0][0] == 0).mean() > 0.02:        # chip hangs off the edge of this scene
            continue
        scl = read_window(item.assets["SCL"].href, bounds, size)
        cloud = float(np.isin(scl, SCL_CLOUD_CLASSES).mean())
        if cloud > args.max_chip_cloud:
            continue
        rgb = np.stack([np.clip(r / 0.3, 0, 1) for _, r in bands], axis=-1)   # 0.3 reflectance = white
        return dict(rgb=(rgb * 255).astype("uint8"), crs=crs, bounds=bounds,
                    date=item.datetime.date().isoformat(), cloud=cloud, item_id=item.id)
    return None


# ----------------------------------------------------------------------------
# 4. Mask + per-chip processing
# ----------------------------------------------------------------------------
def make_mask(mining, sindex, crs, bounds, size):
    tf = Transformer.from_crs(crs.to_wkt(), "EPSG:4326", always_xy=True)
    xs = [bounds[0], bounds[2], bounds[0], bounds[2]]
    ys = [bounds[1], bounds[1], bounds[3], bounds[3]]
    lons, lats = tf.transform(xs, ys)
    idx = sindex.query(box(min(lons), min(lats), max(lons), max(lats)), predicate="intersects")
    mask = np.zeros((size, size), dtype="uint8")
    if len(idx) == 0:
        return mask
    sub = mining.iloc[idx].to_crs(crs.to_wkt())
    transform = transform_from_bounds(*bounds, size, size)
    return rasterize([(g, 1) for g in sub.geometry if g is not None and not g.is_empty],
                     out_shape=(size, size), transform=transform, fill=0,
                     dtype="uint8", all_touched=False)


def process_chip(rec, ctx):
    cid = rec["chip_id"]
    meta_path = ctx["meta_dir"] / f"{cid}.json"
    if meta_path.exists():
        return "cached", None
    args = ctx["args"]
    try:
        got = fetch_chip(rec["lon"], rec["lat"], args)
        if got is None:
            return "fail", "no_clear_scene"
        mask = make_mask(ctx["mining"], ctx["sindex"], got["crs"], got["bounds"], args.chip_px)
        mine_px = int(mask.sum())
        if rec["kind"] == "mine" and mine_px < args.min_mine_px:
            return "fail", "too_little_mining_in_chip"
        if rec["kind"] == "neg" and mine_px > 0:
            return "fail", "negative_chip_contains_mining"
        Image.fromarray(got["rgb"]).save(ctx["img_dir"] / f"{cid}.png")
        Image.fromarray(mask).save(ctx["mask_dir"] / f"{cid}.png")
        meta = dict(chip_id=cid, kind=rec["kind"], split=rec["split"], lon=rec["lon"], lat=rec["lat"],
                    date=got["date"], scene_id=got["item_id"], scene_cloud_in_chip=round(got["cloud"], 4),
                    mine_pixels=mine_px, mine_fraction=round(mine_px / mask.size, 6),
                    crs=got["crs"].to_string(), bounds=list(got["bounds"]),
                    image=f"images/{cid}.png", mask=f"masks/{cid}.png")
        meta_path.write_text(json.dumps(meta))      # written last = "this chip is done" marker
        return "ok", meta
    except Exception as e:
        return "fail", f"{type(e).__name__}: {e}"


# ----------------------------------------------------------------------------
# 5. Manifest + Hugging Face dataset
# ----------------------------------------------------------------------------
def write_manifest(out):
    metas = [json.loads(p.read_text()) for p in sorted((out / "meta").glob("*.json"))]
    if not metas:
        raise SystemExit("No finished chips found. Nothing to build.")
    man = pd.DataFrame(metas)
    man.to_csv(out / "manifest.csv", index=False)
    return man


def build_hf(out, push_to_hub=None):
    from datasets import Dataset, DatasetDict
    from datasets import Image as HFImage
    man = write_manifest(out)
    splits = {}
    for s in ("train", "val", "test"):
        r = man[man.split == s]
        if r.empty:
            print(f"  note: split '{s}' is empty")
            continue
        ds = Dataset.from_dict({
            "image": [str(out / p) for p in r.image],
            "label": [str(out / p) for p in r["mask"]],
            "chip_id": r.chip_id.tolist(), "kind": r.kind.tolist(),
            "lon": r.lon.tolist(), "lat": r.lat.tolist(), "date": r.date.tolist(),
            "mine_fraction": r.mine_fraction.tolist(),
        }).cast_column("image", HFImage()).cast_column("label", HFImage())
        splits[s] = ds
    dd = DatasetDict(splits)
    dd.save_to_disk(str(out / "hf_dataset"))
    print(f"\nHF dataset saved to {out / 'hf_dataset'}")
    print(dd)
    print("Reload with:  from datasets import load_from_disk; ds = load_from_disk('%s')" % (out / "hf_dataset"))
    if push_to_hub:
        dd.push_to_hub(push_to_hub)
        print(f"Pushed to https://huggingface.co/datasets/{push_to_hub}")
    print("\nBy split / kind:")
    print(man.groupby(["split", "kind"]).size().unstack(fill_value=0))


# ----------------------------------------------------------------------------
def in_notebook():
    return "ipykernel" in sys.modules or "google.colab" in sys.modules


def main(argv=None):
    """argv=None reads the command line. In Colab/Jupyter call main([...]) with a list of arguments."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--mining", nargs="+", help="mining polygon file(s): .gpkg/.shp/.geojson (Tang & Werner, Maus, ...)")
    ap.add_argument("--layer", default=None, help="layer name if the file has several layers")
    ap.add_argument("--boundary", default="ne_10m_admin_0_countries.zip", help="Natural Earth countries file (auto-downloaded if missing)")
    ap.add_argument("--out", default="india_mining")
    ap.add_argument("--date-range", default="2019-01-01/2019-12-31",
                    help="Sentinel-2 date window. Maus v2 polygons were drawn on 2019 imagery. "
                         "India's dry season (Nov-Apr) has the fewest clouds, e.g. 2019-11-01/2020-04-30")
    ap.add_argument("--chip-px", type=int, default=512)
    ap.add_argument("--neg-ratio", type=float, default=0.3, help="background-only chips as a fraction of mining chips")
    ap.add_argument("--max-scene-cloud", type=float, default=30.0, help="scene-level cloud %% filter")
    ap.add_argument("--max-chip-cloud", type=float, default=0.10, help="max cloud fraction inside the chip")
    ap.add_argument("--max-scenes-tried", type=int, default=6)
    ap.add_argument("--min-mine-px", type=int, default=50, help="drop mining chips with fewer mining pixels")
    ap.add_argument("--limit", type=int, default=None, help="only build this many mining chips (test run)")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--hf-only", action="store_true", help="skip downloading; rebuild manifest + HF dataset from finished chips")
    ap.add_argument("--push-to-hub", default=None, help="e.g. yourname/india-mining-footprint (needs huggingface-cli login)")
    args, unknown = ap.parse_known_args(argv)
    if unknown and not in_notebook():
        ap.error(f"unrecognized arguments: {' '.join(unknown)}")   # inside Jupyter/Colab the injected '-f kernel.json' is ignored

    out = Path(args.out)
    for d in ("images", "masks", "meta"):
        (out / d).mkdir(parents=True, exist_ok=True)

    if args.hf_only:
        build_hf(out, args.push_to_hub)
        return
    if not args.mining:
        ap.error("--mining is required (unless --hf-only)")

    rng = random.Random(args.seed)
    india = load_india(args.boundary)
    india_geom = shapely.union_all(india.geometry.values)
    mining = load_mining(args.mining, india_geom, args.layer)
    sindex = mining.sindex

    chip_m = args.chip_px * PIXEL_M
    pos = plan_mine_chips(mining, chip_m, dedupe_m=0.4 * chip_m)
    print(f"Planned mining chips: {len(pos):,}")
    if args.limit and len(pos) > args.limit:
        pos = rng.sample(pos, args.limit)
    neg = plan_negative_chips(india_geom, mining, int(len(pos) * args.neg_ratio), rng)
    print(f"Planned background chips: {len(neg):,}")

    recs = pos + neg
    for r in recs:
        r["chip_id"] = f"{r['kind']}_{r['lon']:.4f}_{r['lat']:.4f}"
        r["split"] = assign_split(r["lon"], r["lat"])
    rng.shuffle(recs)

    ctx = dict(args=args, mining=mining, sindex=sindex, meta_dir=out / "meta",
               img_dir=out / "images", mask_dir=out / "masks")
    stats, failures = defaultdict(int), []
    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        futs = {ex.submit(process_chip, r, ctx): r for r in recs}
        for f in tqdm(as_completed(futs), total=len(futs), desc="chips"):
            status, info = f.result()
            stats[status] += 1
            if status == "fail":
                failures.append(dict(chip_id=futs[f]["chip_id"], reason=info))
    print(dict(stats))
    if failures:
        pd.DataFrame(failures).to_csv(out / "failed.csv", index=False)
        print(f"{len(failures)} chips failed (reasons in {out / 'failed.csv'}). Re-run the same command to retry them.")

    build_hf(out, args.push_to_hub)
    print("\nLabels: 0 = background, 1 = mining.  NOTE: source polygons are incomplete, so some "
          "unlabeled mines will sit inside 'background' pixels (label noise).")


# =====================================================================================
#  RUNNER: download -> unpack -> find polygons -> build dataset -> zip -> download
# =====================================================================================
import shutil, tarfile, urllib.request, zipfile
import requests


def get_boundary(path="/content/ne_10m_admin_0_countries.zip"):
    """India's outline (Natural Earth). Tries two mirrors."""
    if Path(path).exists():
        return path
    for url in (NE_URL, "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip"):
        try:
            print("Downloading country boundaries:", url)
            urllib.request.urlretrieve(url, path)
            if zipfile.is_zipfile(path):
                return path
        except Exception as e:
            print("  failed:", e)
    raise SystemExit("Could not download the country-boundary file. Download ne_10m_admin_0_countries.zip "
                     "from naturalearthdata.com, upload it to /content, and run this cell again.")


def download_zenodo(record_id, dest):
    dest.mkdir(parents=True, exist_ok=True)
    r = requests.get(f"https://zenodo.org/api/records/{record_id}", timeout=60)
    r.raise_for_status()
    files = r.json().get("files", [])
    if not files:
        raise SystemExit(f"Zenodo record {record_id} lists no downloadable files.")
    for f in files:
        name = Path(f.get("key") or f.get("filename") or "file").name
        links = f.get("links") or {}
        url = links.get("self") or links.get("download")
        if not url:
            print("  skipping (no link):", name)
            continue
        target, size = dest / name, f.get("size", 0)
        if target.exists() and size and target.stat().st_size == size:
            print("  already downloaded:", name)
            continue
        print(f"  downloading {name} ({size / 1e6:.0f} MB) ...")
        with requests.get(url, stream=True, timeout=120) as resp:
            resp.raise_for_status()
            with open(target, "wb") as fh:
                for chunk in resp.iter_content(1 << 20):
                    fh.write(chunk)


def extract_all(folder):
    for p in sorted(folder.rglob("*")):
        if not p.is_file():
            continue
        n = p.name.lower()
        dest = p.parent / (p.name + "_extracted")
        if dest.exists() or not n.endswith((".zip", ".tar", ".tar.gz", ".tgz", ".rar")):
            continue
        print("  extracting", p.name)
        if n.endswith(".zip"):
            zipfile.ZipFile(p).extractall(dest)
        elif n.endswith((".tar", ".tar.gz", ".tgz")):
            tarfile.open(p).extractall(dest)
        else:  # .rar
            exe = shutil.which("unrar") or shutil.which("unar")
            if exe is None:
                subprocess.run("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y unrar > /dev/null 2>&1 "
                               "|| apt-get -qq install -y unar > /dev/null 2>&1", shell=True)
                exe = shutil.which("unrar") or shutil.which("unar")
            if exe is None:
                raise SystemExit("Couldn't install an unrar tool. Run  !apt-get install -y unrar  in a cell, then re-run.")
            dest.mkdir(parents=True)
            if os.path.basename(exe) == "unrar":
                subprocess.run([exe, "x", "-o+", "-y", str(p), str(dest) + "/"], check=True, stdout=subprocess.DEVNULL)
            else:
                subprocess.run([exe, "-o", str(dest), "-f", str(p)], check=True, stdout=subprocess.DEVNULL)


def find_polygon_layers(folder, min_features=500):
    import pyogrio
    cands = []
    for p in sorted(folder.rglob("*")):
        if p.suffix.lower() not in (".gpkg", ".shp", ".geojson"):
            continue
        try:
            for name, _ in pyogrio.list_layers(str(p)):
                info = pyogrio.read_info(str(p), layer=name)
                gt = str(info.get("geometry_type") or "unknown").lower()
                if "polygon" in gt or "unknown" in gt:
                    cands.append((p, name, int(info.get("features", -1))))
        except Exception as e:
            print("  skipping", p.name, "->", e)
    if not cands:
        raise SystemExit("No polygon map files (.gpkg/.shp/.geojson) found after unpacking. "
                         "List them with:  !find /content/tang_werner_raw | head -50")
    big = [c for c in cands if c[2] >= min_features]
    chosen = big or [max(cands, key=lambda c: c[2])]
    print("Polygon layers found:")
    for p, name, n in cands:
        print(f"  {'USING' if (p, name, n) in chosen else 'skip '}  {p.relative_to(folder)}  [{name}]  {n:,} features")
    return chosen


def merge_to_gpkg(chosen, out_path):
    frames = []
    for p, layer, _ in chosen:
        g = gpd.read_file(p, layer=layer)
        if g.crs is None:
            b = g.total_bounds
            if abs(b).max() > 360:
                raise SystemExit(f"{p.name} has no coordinate system and its coordinates are not lat/lon.")
            g = g.set_crs(4326)
        frames.append(g[["geometry"]].to_crs(4326))
    merged = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry", crs=4326)
    merged.to_file(out_path, driver="GPKG")
    print(f"Merged {len(merged):,} polygons -> {out_path}")
    return str(out_path)


def run_all():
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    out = "/content/drive/MyDrive/india_mining" if SAVE_TO_DRIVE else "/content/india_mining"
    raw = Path("/content/tang_werner_raw")

    print("\n[1/5] Downloading Tang & Werner mining polygons from Zenodo ...")
    download_zenodo(ZENODO_RECORD, raw)
    print("\n[2/5] Unpacking ...")
    extract_all(raw)
    print("\n[3/5] Finding the polygon file ...")
    merged = merge_to_gpkg(find_polygon_layers(raw), Path("/content/mining_polygons_all.gpkg"))
    boundary = get_boundary()

    print("\n[4/5] Building the India dataset (images + masks + Hugging Face dataset) ...")
    argv = ["--mining", merged, "--boundary", boundary, "--out", out,
            "--date-range", DATE_RANGE, "--workers", str(WORKERS)]
    if LIMIT:
        argv += ["--limit", str(LIMIT)]
    try:
        main(argv)
    except SystemExit as e:
        print("\nStopped:", e)

    out_p = Path(out)
    if not (out_p / "manifest.csv").exists():
        print("\nNo chips were produced, so there is nothing to zip.")
        if (out_p / "failed.csv").exists():
            print("Why chips failed:\n", pd.read_csv(out_p / "failed.csv")["reason"].value_counts().head(10))
        return

    print("\n[5/5] Zipping into ONE file ...")
    zip_path = shutil.make_archive("/content/india_mining_dataset", "zip", root_dir=out)
    print(f"Created {zip_path}  ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
    print("Inside: images/  masks/  manifest.csv  hf_dataset/   (labels: 0 = background, 1 = mining)")
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print("Not running in Colab. The zip is at", zip_path)


run_all()

Installing: ['pystac-client', 'planetary-computer']

[1/5] Downloading Tang & Werner mining polygons from Zenodo ...
  downloading Supplementary 1：mine area polygons.rar (104 MB) ...

[2/5] Unpacking ...
  extracting Supplementary 1：mine area polygons.rar

[3/5] Finding the polygon file ...
Polygon layers found:
  USING  Supplementary 1：mine area polygons.rar_extracted/Supplementary 1：mine area polygons/74548 mine polygons/74548_projected.shp  [74548_projected]  74,548 features
  USING  Supplementary 1：mine area polygons.rar_extracted/Supplementary 1：mine area polygons/Artisanal and small-scale mine/artisanal and small-scale mine.shp  [artisanal and small-scale mine]  4,058 features
Merged 78,606 polygons -> /content/mining_polygons_all.gpkg

[4/5] Building the India dataset (images + masks + Hugging Face dataset) ...
Reading /content/mining_polygons_all.gpkg ...
  78,606 features
Mining polygons inside/touching India: 1,808
Planned mining chips: 772
Planned background chips: 60


chips:  71%|███████   | 184/260 [01:23<00:30,  2.49it/s]/usr/local/lib/python3.13/dist-packages/rasterio/features.py:381: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _rasterize(
chips: 100%|██████████| 260/260 [01:54<00:00,  2.28it/s]


{'ok': 254, 'fail': 6}
6 chips failed (reasons in /content/india_mining/failed.csv). Re-run the same command to retry them.


Saving the dataset (0/1 shards):   0%|          | 0/151 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]


HF dataset saved to /content/india_mining/hf_dataset
DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'chip_id', 'kind', 'lon', 'lat', 'date', 'mine_fraction'],
        num_rows: 151
    })
    val: Dataset({
        features: ['image', 'label', 'chip_id', 'kind', 'lon', 'lat', 'date', 'mine_fraction'],
        num_rows: 32
    })
    test: Dataset({
        features: ['image', 'label', 'chip_id', 'kind', 'lon', 'lat', 'date', 'mine_fraction'],
        num_rows: 71
    })
})
Reload with:  from datasets import load_from_disk; ds = load_from_disk('/content/india_mining/hf_dataset')

By split / kind:
kind   mine  neg
split           
test     62    9
train   109   42
val      24    8

Labels: 0 = background, 1 = mining.  NOTE: source polygons are incomplete, so some unlabeled mines will sit inside 'background' pixels (label noise).

[5/5] Zipping into ONE file ...
Created /content/india_mining_dataset.zip  (231 MB)
Inside: images/  masks/  manifest.csv  hf_dataset/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>